In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, train_test_split, cross_val_score
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import RFE

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

from xgboost import XGBClassifier
import optuna

RANDOM_STATE = 42

# LOAD
df = pd.read_csv("/kaggle/input/...csv")

# TARGET
df['Attrition'] = df['Attrition'].map({'Yes':1, 'No':0})

# DROP USELESS
df.drop(["EmployeeCount","EmployeeNumber","Over18","StandardHours"], axis=1, inplace=True)

# FEATURES
X = df.drop("Attrition", axis=1)
y = df["Attrition"]

cat_cols = X.select_dtypes(include="object").columns
num_cols = X.select_dtypes(include=np.number).columns

# PREPROCESS
preprocessor = ColumnTransformer([
    ("num", RobustScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
])

# --------------------
# BASELINE MODEL
# --------------------
baseline = Pipeline([
    ("prep", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

baseline.fit(X, y)
print("Baseline F1:", np.mean(cross_val_score(baseline, X, y, cv=5, scoring="f1")))

# --------------------
# RFE FEATURE SELECTION
# --------------------
rfe_model = LogisticRegression(max_iter=1000)

rfe = RFE(rfe_model, n_features_to_select=15)
X_rfe = rfe.fit_transform(pd.get_dummies(X), y)

# --------------------
# OPTUNA (F1 OPTIMIZATION)
# --------------------
def objective(trial):

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 600),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1, 5)
    }

    model = ImbPipeline([
        ("prep", preprocessor),
        ("smote", SMOTE(random_state=RANDOM_STATE)),
        ("model", XGBClassifier(
            **params,
            random_state=RANDOM_STATE,
            eval_metric="logloss"
        ))
    ])

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

    scores = cross_val_score(model, X, y, cv=cv, scoring="f1")
    return scores.mean()

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30)

# --------------------
# FINAL MODEL
# --------------------
best_params = study.best_params

final_model = ImbPipeline([
    ("prep", preprocessor),
    ("smote", SMOTE(random_state=RANDOM_STATE)),
    ("model", XGBClassifier(
        **best_params,
        random_state=RANDOM_STATE,
        eval_metric="logloss"
    ))
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

final_model.fit(X_train, y_train)

# --------------------
# EVALUATION
# --------------------
y_pred = final_model.predict(X_test)

print("F1:", f1_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))